In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from joblib import dump

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import mlflow
import mlflow.sklearn

# Notebook Overview & Library Imports
This notebook trains and compares intrusion detection classifiers on a filtered subset of the UNSW_NB15 dataset (Normal vs selected attack categories). Each section is preceded by an explanation covering:
- What the code does
- Why it is needed
- Key machine learning / data concepts
- Potential caveats & improvement ideas

Below we import:
- `numpy`, `pandas`: numerical & tabular data handling.
- `seaborn`, `matplotlib`: visualization.
- `sklearn` modules: preprocessing, modeling (RandomForest), evaluation metrics, and `Pipeline` composition.
- `xgboost`, `lightgbm`: gradient boosting libraries (often strong tabular baselines).
- `joblib`: persisting trained artifacts for reuse.
- `mlflow`: experiment tracking (parameters, metrics, artifacts) for reproducibility and comparative analysis.

Design notes:
- Centralized imports keep dependency surface visible.
- Using multiple model families provides bias/variance & algorithmic diversity.
- MLflow facilitates later aggregation of results & auditability.


# Reproducibility & Random Seeds
We set a global `RANDOM_STATE` to make the experiment as deterministic as possible. This helps:
- Recreating hyperparameter search results (same random draws in `RandomizedSearchCV`).
- Ensuring model initialization (where supported) is constant across runs.
- Providing stable comparison metrics for the thesis.

Limitations:
- Some algorithms (multi-threaded LightGBM / XGBoost) can retain slight nondeterminism due to parallelism.
- Full determinism would also require controlling Python hash seed (`PYTHONHASHSEED`) and potentially limiting thread counts.

Why keep it: Transparency, audit trail, and easier debugging.
When you might drop it: Large-scale randomized ensembling where diversity is desired.


In [ ]:
# --- Reproducibility ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# --- Load Dataset ---
df_train = pd.read_parquet("./datasets/UNSW_NB15_training-set.parquet")
df_test = pd.read_parquet("./datasets/UNSW_NB15_testing-set.parquet")

# Data Loading (Training & Test Sets)
We load pre-split Parquet files for training and testing.
Why Parquet:
- Columnar + compressed → faster IO and lower disk footprint than CSV.
- Preserves dtypes (avoids implicit type guessing bugs).

Assumptions & Rationale:
- Split performed upstream → prevents train/test leakage during preprocessing.
- Using separate files avoids accidental reshuffling that could cause optimistic metrics.

Caveats:
- Always verify schema consistency between train and test (same columns, compatible dtypes).
- If dataset versioning matters, log a hash (e.g., md5) to MLflow for auditability.


In [ ]:
def preprocess_data(df):
    """Preprocess dataset and return X, y."""
    df_filtered = df[df['attack_cat'].isin(['Worms', 'Backdoor']) | (df['label'] == 0)].copy()
    df_filtered['attack_label'] = df_filtered['attack_cat'].fillna('Normal')
    df_filtered = df_filtered.drop(columns=[c for c in ['id', 'label', 'attack_cat'] if c in df_filtered])
    
    # Features
    X = df_filtered.drop(columns=['attack_label'])
    # Target
    y = df_filtered['attack_label']
    return X, y

# Preprocessing Function Definition
Purpose:
- Filter to focus classification on two attack categories ("Worms", "Backdoor") plus normal traffic, reducing multi-class complexity.
- Create a clean target label `attack_label` consolidating Normal vs chosen attacks.
- Drop potential leakage / ID columns (`id`, original `label`, raw `attack_cat`).

Why Filtering:
- Narrow scope can improve minority class performance analysis and interpretability.
- Reduces class imbalance severity versus full UNSW_NB15 taxonomy.

Concepts:
- Data Leakage: Removing original numeric `label` prevents accidentally giving away the answer.
- Copying DataFrame avoids chained assignment side-effects.

Outputs:
- `X`: Feature matrix without target.
- `y`: Clean multi-class target (strings), ready for encoding.

Improvement Ideas:
- Parameterize the kept attack categories for reuse.
- Add validation asserts (e.g., ensure all expected classes present post-filter).


In [ ]:
X_train_raw, y_train = preprocess_data(df_train)
X_test_raw, y_test = preprocess_data(df_test)

print("--- Class Distribution in Training Set ---")
print(y_train.value_counts(), "\n" + "-"*40)
print("--- Class Distribution in Test Set ---")
print(y_test.value_counts(), "\n" + "-"*40)


# Apply Preprocessing & Inspect Class Balance
Here we:
1. Generate raw features / targets for train and test using the preprocessing function.
2. Print class distributions to assess imbalance.

Why It Matters:
- Class imbalance biases accuracy; motivates using F1 (weighted + macro) later.
- Early visibility helps decide on strategies (class weighting, sampling, thresholding).

Concepts:
- Support: Number of instances per class, used in weighted metrics.
- If a class is extremely rare, cross-validation folds might miss it → consider stratification if splitting again.

Next Steps:
- Proceed to feature encoding aware of imbalance context.


# One-Hot Encoding & Column Alignment
What:
- Convert categorical feature columns into binary indicator (0/1) columns using `pd.get_dummies` (one-hot encoding).
- Align train and test feature matrices so they share identical column sets and ordering.

Why Needed:
- Most tree/boosting models need numeric inputs; categorical strings are unsupported directly here (unless using native categorical handling, which we are not enabling).
- Categories present in train but absent in test (or vice versa) would cause shape mismatch or silent misinterpretation without alignment.

How Alignment Works:
- `X_train.align(X_test, join="left", axis=1, fill_value=0)` reindexes `X_test` to the columns of `X_train` (left join), inserting 0 for any unseen category in test.
- Ensures consistent feature space → model interprets each column identically between fit and predict.

Risks / Caveats:
- High cardinality categories inflate dimensionality (sparsity, memory cost).
- If test introduces a novel category absent in train, it is silently ignored (all-zero column) → model cannot learn its effect; consider monitored logging of unseen categories.

Alternatives / Improvements:
- Use `sklearn` `OneHotEncoder(handle_unknown='ignore')` within a `ColumnTransformer` to avoid manual alignment.
- For large cardinality: target encoding, hashing trick, or LightGBM native categorical support.


In [ ]:
# One-hot encoding + alignment
X_train = pd.get_dummies(X_train_raw)
X_test = pd.get_dummies(X_test_raw)
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# Label Encoding the Target
What:
- `LabelEncoder` maps textual class labels (e.g., `Normal`, `Backdoor`, `Worms`) to integer indices (0..K-1).

Why:
- Many scikit-learn classifiers and metrics expect integer-encoded targets.
- Provides a compact representation; needed for `classification_report` with `target_names` mapping back to human-readable labels.

Why Not One-Hot for Target:
- Multi-class single-label problem: exactly one class per sample. One-hot target adds unnecessary dimensionality and conversions.

Persistence:
- We save the fitted encoder to reconstruct human-readable predictions during inference / reporting.

Caveats:
- Do not use `LabelEncoder` on nominal feature columns (would inject artificial ordinality). Use one-hot or other encodings there.
- If class set changes in future data, the saved encoder must be updated carefully (versioning advisable).


In [ ]:
# Label encoding
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)
class_labels = le.classes_

# Model Set & Hyperparameter Search Space
We compare three complementary tree-based model families:
- RandomForest: Bagging ensemble → robust, lower variance, easier to tune.
- XGBoost: Gradient boosting with advanced regularization; strong tabular performance.
- LightGBM: Histogram-based gradient boosting → faster training on large, sparse features.

Why Multiple Models:
- Different bias/variance profiles; some may better capture minority class patterns.
- Provides empirical justification for final model choice beyond single attempt.

Hyperparameters (brief semantics):
- `n_estimators`: Number of trees (too low → underfit, too high → slower, diminishing returns).
- `max_depth`: Controls tree complexity (regularization).
- `min_samples_split` / `min_samples_leaf` (RF): Prevent overly specific splits.
- `learning_rate` (boosting): Step size shrinkage; lower values need more estimators.
- `subsample` (XGBoost): Stochasticity to reduce overfitting.
- `num_leaves` (LightGBM): Leaf count upper bound; higher → more expressive, risk overfit.

Why RandomizedSearchCV:
- More efficient exploration vs full grid when many combinations exist.
- Allows easy scaling by raising `n_iter` if time permits.

Imbalance Handling:
- `class_weight='balanced'` for models that accept it, weighting minority classes higher.

Potential Enhancements:
- Use distributions (e.g., loguniform) rather than small discrete lists.
- Introduce early stopping for boosting models via validation set.
- Evaluate additional metrics (macro F1) during search via custom scorer.


In [ ]:
# --- Define Models & Params ---
models = {
    'RandomForest': RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    'XGBoost': XGBClassifier(random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='mlogloss'),
    'LightGBM': LGBMClassifier(random_state=RANDOM_STATE, class_weight='balanced')
}

param_grids = {
    'RandomForest': {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
    },
    'XGBoost': {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [5, 10, 15],
        'classifier__learning_rate': [0.05, 0.1, 0.2],
        'classifier__subsample': [0.7, 0.8],
    },
    'LightGBM': {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [10, 20, -1],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__num_leaves': [31, 50],
    }
}

# MLflow Experiment Configuration
We initialize / select an MLflow experiment (`UNSW_NB15_Classification`).

Purpose of Tracking:
- Central store of runs: hyperparameters, metrics, artifacts (confusion matrices, model binaries).
- Reproducibility: Enables later auditing of how a reported score was produced.
- Comparison: Facilitates post-hoc aggregation & ranking across algorithms.

Artifacts Planned:
- Confusion matrices (PNG) per model run.
- Serialized model pipeline (scaler + estimator) per run.
- Later: consolidated comparison CSV & plots.

Good Practices:
- Tag runs with dataset version or git commit (could add `mlflow.set_tag`).
- Avoid logging sensitive data (only derived metrics/artifacts here).


In [ ]:
# --- Training Loop with MLflow ---
best_score = -1
best_model_pipeline = None
best_model_name = ""

mlflow.set_experiment("UNSW_NB15_Classification")

# Training Loop: Hyperparameter Search + Evaluation
Workflow Per Model:
1. Construct a `Pipeline` (scaler + classifier). (Scaler largely neutral for tree models but keeps structure consistent.)
2. Run `RandomizedSearchCV` (3-fold) optimizing weighted F1 to balance class contributions by support.
3. Fit on full training data using best hyperparameters from CV.
4. Predict on held-out test set (true generalization estimate).
5. Compute metrics: weighted F1 + full classification report (precision/recall/F1 per class, macro averages).
6. Log parameters, metrics, confusion matrix, and model to MLflow.
7. Track global best model for downstream persistence.

Key Concepts:
- Weighted F1: Penalizes misclassification of minority classes but still influenced by majority support.
- Confusion Matrix (normalized): Row-wise percentages for interpretability across imbalanced classes.
- Separation of concerns: Search (CV) happens strictly on training set → prevents test leakage.

Potential Optimizations:
- Remove scaler to reduce compute time.
- Introduce early stopping (boosting) using a validation split.
- Parallelize beyond `n_jobs=-1` only if memory permits.


In [ ]:
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        print(f"\n{'='*20}\nTraining {name}...\n{'='*20}")

        pipeline = Pipeline([
            ('scaler', StandardScaler()),  # can be dropped for tree models
            ('classifier', model)
        ])

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grids[name],
            n_iter=10,
            cv=3,
            verbose=1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            scoring='f1_weighted'
        )

        search.fit(X_train, y_train_encoded)
        best_pipeline_for_model = search.best_estimator_
        y_pred_encoded = best_pipeline_for_model.predict(X_test)

        # --- Metrics ---
        current_score = f1_score(y_test_encoded, y_pred_encoded, average='weighted')
        report = classification_report(y_test_encoded, y_pred_encoded, target_names=class_labels, output_dict=True)

        print(f"\n--- Results for {name} ---")
        print("Best Parameters:", search.best_params_)
        print(f"Weighted F1-Score on Test Set: {current_score:.4f}")

        # --- MLflow Logging ---
        mlflow.log_params(search.best_params_)
        mlflow.log_metric("f1_weighted", current_score)

        # Log classification report metrics
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)

        # Log confusion matrix as artifact
        cm = confusion_matrix(y_test_encoded, y_pred_encoded, normalize="true")
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                    xticklabels=class_labels, yticklabels=class_labels)
        plt.title(f'Confusion Matrix - {name}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.tight_layout()
        plt.savefig(f"confusion_matrix_{name}.png")
        mlflow.log_artifact(f"confusion_matrix_{name}.png")

        # Log model itself
        mlflow.sklearn.log_model(best_pipeline_for_model, artifact_path="model")

        # Track best model
        if current_score > best_score:
            best_score = current_score
            best_model_pipeline = best_pipeline_for_model
            best_model_name = name

In [ ]:
# --- Final Save ---
print(f"\n{'='*20}\nOverall Best Model: {best_model_name} (Weighted F1: {best_score:.4f})\n{'='*20}")
dump(best_model_pipeline, f'best_model_pipeline_{best_model_name}.joblib')
dump(X_train.columns, 'model_columns.joblib')
dump(le, 'label_encoder.joblib')
print("Artifacts saved successfully.")

# Persist Best Model & Supporting Artifacts
We save:
- `best_model_pipeline_<Name>.joblib`: End-to-end pipeline (preprocessor stage + model) for inference reuse.
- `model_columns.joblib`: Exact ordered feature columns after one-hot encoding & alignment (guards against schema drift at inference).
- `label_encoder.joblib`: Target encoder to translate numeric predictions back to semantic labels.

Why Persistence Matters:
- Reproducibility: Exact objects used to produce reported metrics.
- Deployment: Avoid re-fitting or re-deriving encodings in production.
- Auditing: Enables later validation / drift analysis.

Caveats:
- Any future change in preprocessing must version bump artifacts.
- Consider storing a hash of feature column list to detect mismatches at serve time.


In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score

# --- Model Comparison & Visualization (Post-Training Analysis) ---

# This cell pulls trained runs from MLflow, loads each model artifact,
# evaluates uniformly on the held-out test set, and produces comparison
# tables & plots for thesis reporting.


EXPERIMENT_NAME = "UNSW_NB15_Classification"

# 1. Fetch MLflow runs
try:
    runs_df = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME])
except Exception as e:
    print(f"Could not retrieve MLflow runs: {e}")
    runs_df = pd.DataFrame()

if runs_df.empty:
    print("No runs found. Execute training loop first.")
else:
    # Keep only finished runs that logged a model
    core_cols = ['run_id', 'metrics.f1_weighted']
    runs_df = runs_df.sort_values("metrics.f1_weighted", ascending=False)
    
    # 2. Load each model artifact and re-evaluate (ensures consistent evaluation code path)
    model_results = []
    loaded_models = {}
    
    # We will only compare models whose names were in original 'models' dict
    candidate_runs = runs_df[runs_df['tags.mlflow.runName'].isin(models.keys())]
    
    for _, row in candidate_runs.iterrows():
        run_id = row.run_id
        model_name = row['tags.mlflow.runName']
        try:
            m = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
            loaded_models[model_name] = m
        except Exception as e:
            print(f"Skip {model_name} (run {run_id}) - load error: {e}")

    # 3. Evaluate uniformly
    # Ensure we only work with classes seen in training label encoder (class_labels)
    class_list = list(class_labels)
    class_index = {c: i for i, c in enumerate(class_list)}

    y_true = y_test.replace({c: c for c in class_list})
    y_true_encoded = y_test_encoded  # already aligned with 'le'
    
    # Binarize for PR curves
    y_test_bin = label_binarize(y_true_encoded, classes=range(len(class_list)))
    
    per_model_probs = {}
    
    for name_, mdl in loaded_models.items():
        y_pred = mdl.predict(X_test)
        y_prob = None
        if hasattr(mdl, "predict_proba"):
            try:
                y_prob = mdl.predict_proba(X_test)
            except Exception:
                y_prob = None
        report_local = classification_report(
            y_true_encoded,
            y_pred,
            target_names=class_list,
            output_dict=True
        )
        cm_local = confusion_matrix(y_true_encoded, y_pred, normalize='true')
        weighted_f1 = report_local['weighted avg']['f1-score']
        macro_f1 = report_local['macro avg']['f1-score']
        per_model_probs[name_] = y_prob
        
        model_results.append({
            "model": name_,
            "weighted_f1": weighted_f1,
            "macro_f1": macro_f1,
            **{f"f1_{cls}": report_local[cls]['f1-score'] for cls in class_list},
            **{f"precision_{cls}": report_local[cls]['precision'] for cls in class_list},
            **{f"recall_{cls}": report_local[cls]['recall'] for cls in class_list},
        })
    
    if not model_results:
        print("No model results computed.")
    else:
        results_df = pd.DataFrame(model_results).sort_values("weighted_f1", ascending=False)
        print("\n=== Summary Metrics (Higher is Better) ===")
        display(results_df.set_index("model"))
        
        # 4. Plot: Weighted & Macro F1
        plt.figure(figsize=(6,4))
        sns.barplot(data=results_df, x='model', y='weighted_f1', palette='Blues_d')
        plt.title('Weighted F1 by Model')
        plt.ylabel('Weighted F1')
        plt.ylim(0, 1.05)
        for i,v in enumerate(results_df['weighted_f1']):
            plt.text(i, v+0.005, f"{v:.4f}", ha='center', fontsize=9)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(6,4))
        sns.barplot(data=results_df, x='model', y='macro_f1', palette='Greens_d')
        plt.title('Macro F1 by Model')
        plt.ylabel('Macro F1')
        plt.ylim(0, 1.05)
        for i,v in enumerate(results_df['macro_f1']):
            plt.text(i, v+0.005, f"{v:.4f}", ha='center', fontsize=9)
        plt.tight_layout()
        plt.show()
        
        # 5. Per-Class F1 grouped bar
        melted_f1 = results_df.melt(
            id_vars="model",
            value_vars=[f"f1_{c}" for c in class_list],
            var_name="class",
            value_name="f1"
        )
        melted_f1['class'] = melted_f1['class'].str.replace("f1_", "")
        plt.figure(figsize=(8,4))
        sns.barplot(data=melted_f1, x='class', y='f1', hue='model')
        plt.title('Per-Class F1 by Model')
        plt.ylim(0, 1.05)
        plt.tight_layout()
        plt.show()
        
        # 6. Confusion Matrices
        n_models = len(loaded_models)
        fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4), squeeze=False)
        for ax, (name_, mdl) in zip(axes[0], loaded_models.items()):
            y_pred = mdl.predict(X_test)
            cm_m = confusion_matrix(y_true_encoded, y_pred, normalize='true')
            sns.heatmap(cm_m, annot=True, fmt=".2f", cmap="Blues",
                        xticklabels=class_list, yticklabels=class_list, ax=ax)
            ax.set_title(f"Confusion - {name_}")
            ax.set_xlabel("Predicted")
            ax.set_ylabel("True")
        plt.tight_layout()
        plt.show()
        
        # 7. Precision-Recall Curves (One-vs-Rest)
        # Only if probabilities available for all models
        if all(per_model_probs[m] is not None for m in loaded_models):
            plt.figure(figsize=(6*len(class_list), 4))
            for c_idx, cls in enumerate(class_list):
                plt.subplot(1, len(class_list), c_idx+1)
                for name_, probs in per_model_probs.items():
                    precision, recall, _ = precision_recall_curve(y_test_bin[:, c_idx], probs[:, c_idx])
                    ap = average_precision_score(y_test_bin[:, c_idx], probs[:, c_idx])
                    plt.plot(recall, precision, label=f"{name_} (AP={ap:.3f})")
                plt.title(f"PR Curve - {cls}")
                plt.xlabel("Recall")
                plt.ylabel("Precision")
                plt.xlim(0,1)
                plt.ylim(0,1.05)
                plt.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
        else:
            print("Skipping PR curves (missing predict_proba on at least one model).")
        
        # 8. Export comparison artifacts
        results_df.to_csv("model_comparison_metrics.csv", index=False)
        print("Saved: model_comparison_metrics.csv")

        # Optional: Save figures programmatically (re-run minimal plotting & save)
        def save_barplot(df, metric, fname, title):
            plt.figure(figsize=(6,4))
            sns.barplot(data=df, x='model', y=metric, palette='viridis')
            plt.title(title)
            plt.ylim(0,1.05)
            for i,v in enumerate(df[metric]):
                plt.text(i, v+0.005, f"{v:.4f}", ha='center', fontsize=9)
            plt.tight_layout()
            plt.savefig(fname, dpi=150)
            plt.close()

        save_barplot(results_df, "weighted_f1", "weighted_f1_models.png", "Weighted F1 by Model")
        save_barplot(results_df, "macro_f1", "macro_f1_models.png", "Macro F1 by Model")
        melted_f1.to_csv("per_class_f1_long.csv", index=False)

        print("Saved: weighted_f1_models.png, macro_f1_models.png, per_class_f1_long.csv")

        # Log comparison to MLflow (optional)
        with mlflow.start_run(run_name="Model_Comparison_Aggregation", nested=True):
            mlflow.log_artifact("model_comparison_metrics.csv")
            if os.path.exists("weighted_f1_models.png"):
                mlflow.log_artifact("weighted_f1_models.png")
            if os.path.exists("macro_f1_models.png"):
                mlflow.log_artifact("macro_f1_models.png")
            if os.path.exists("per_class_f1_long.csv"):
                mlflow.log_artifact("per_class_f1_long.csv")
            for _, r in results_df.iterrows():
                mlflow.log_metric(f"{r['model']}_weighted_f1", r['weighted_f1'])
                mlflow.log_metric(f"{r['model']}_macro_f1", r['macro_f1'])
        print("Comparison artifacts logged to MLflow.")

# Thesis usage hints:
# - Use results_df table for quantitative comparison.
# - Use confusion matrices for error mode discussion.
# - Use PR curves to analyze minority class (Worms) performance.
# - Weighted vs Macro F1 discussion highlights class imbalance handling.

# Post-Training Comparative Analysis & Visualization
Objectives:
- Uniformly re-evaluate all tracked MLflow model runs on the same test set to ensure consistent metric computation.
- Produce summary and diagnostic visualizations for thesis/reporting.

Steps:
1. Query MLflow for runs under experiment name.
2. Load each model artifact (ensures evaluation code path consistency vs relying on logged metrics alone).
3. Compute per-model classification report → weighted & macro F1 plus per-class precision/recall/F1.
4. Visualize: bar charts (weighted & macro F1), grouped per-class F1, confusion matrices, and (optionally) precision–recall curves.
5. Persist comparison artifacts (CSV + PNGs) and optionally log them back to MLflow in a nested run.

Key Metric Concepts:
- Weighted F1: Support-weighted performance; can mask minority degradation.
- Macro F1: Treats each class equally → fairness / minority sensitivity.
- PR Curves: More informative under class imbalance than ROC; AP (Average Precision) summarizes area.

Why Reload Models:
- Guards against inconsistencies if logging logic changes later.
- Enables extending evaluation (e.g., new metrics) without retraining.

Further Enhancements:
- Add bootstrapped confidence intervals for metrics.
- Add calibration curves if probability estimates will drive downstream risk scoring.
- Integrate SHAP / feature importance plots for interpretability section.
